# Setup Path

In [1]:
import sys
sys.path.insert(0, "..")

# Import Library

In [2]:
import pandas as pd
from src.db import get_engine

pd.set_option('display.max_rows', 150)
engine = get_engine()

# Validasi Ingestion — Deka Norm Database

Notebook ini digunakan untuk memverifikasi bahwa data yang masuk ke database sudah bersih dan lengkap.

## 1. Ringkasan per Project

In [3]:
query = """
SELECT 
    p.project_id,
    p.year,
    p.category,
    p.sub_category,
    p.detail_product,
    COUNT(DISTINCT r.respondent_id) AS n_respondents,
    COUNT(DISTINCT r.variable_name) AS n_variables,
    COUNT(*)                        AS n_responses
FROM responses r
JOIN projects p ON r.project_id = p.project_id
GROUP BY p.project_id, p.year, p.category, p.sub_category, p.detail_product
ORDER BY p.year, p.project_id;
"""

df_summary = pd.read_sql(query, engine)
df_summary

,project_id,year,category,sub_category,detail_product,n_respondents,n_variables,n_responses
0,DKT-202501-0007,2025,Food,Noodles,Instant Noodle,150,12,2700
1,DKT-202504-0044,2025,Food,Seasoning,Beef Boullion,370,12,4440
2,DKT-202506-0069,2025,Food,Seasoning,Beef Boullion,360,11,3960
3,DKT-202507-0071,2025,Personal Care,Face Care,Blurring Suncreen,30,31,930
4,DKT-202509-0085,2025,Food,Biscuit,Biscuits with Filling,150,25,3750
5,DKT-202509-0090,2025,Confectionary,Candy,Coffee Candy,178,22,1960
6,DKT-202509-0097,2025,Beverages,Coffee,Coffee Instant,121,18,2178
7,DKT-202510-0105,2025,Food,Biscuit,Brownies Crispy,50,20,1000
8,DKT-202510-0111,2025,Food,Seasoning,Rendang Seasoning,51,15,765
9,DKT-202511-0113,2025,Food,Noodles,Instant Noodle,97,17,1649


## 2. Daftar Variable Name
Cek konsistensi penamaan variabel setelah canonicalization.

In [4]:
query = """
SELECT 
    variable_name,
    COUNT(DISTINCT project_id) AS n_projects,
    COUNT(*)                   AS n_responses
FROM responses
GROUP BY variable_name
ORDER BY n_projects DESC, variable_name;
"""

df_vars = pd.read_sql(query, engine)
print(f"Total unique variable_name: {len(df_vars)}")
df_vars

Total unique variable_name: 119


,variable_name,n_projects,n_responses
0,aftertaste,15,2282
1,overall taste,15,2282
2,aroma,14,2132
3,overall liking,14,2084
4,appearance,13,1552
5,color,13,1474
6,purchase intent w/o price,13,1952
7,sweetness,11,1849
8,saltiness,7,1313
9,savoriness,7,1313


## 3. Validasi Score Normalized
Memastikan score_normalized berada di rentang 0–1.

In [5]:
query = """
SELECT 
    MIN(score_normalized)  AS min_norm,
    MAX(score_normalized)  AS max_norm,
    AVG(score_normalized)  AS avg_norm,
    COUNT(*)               AS total_responses
FROM responses;
"""

df_score_check = pd.read_sql(query, engine)
df_score_check

,min_norm,max_norm,avg_norm,total_responses
0,0.0,1.0,0.808221,35341


## 4. Norm Score (Parameter Utama)
Norm score untuk `overall liking` dan `purchase intent w/o price` per category dan sub_category.

In [6]:
query = """
SELECT
    p.category,
    p.sub_category,
    r.variable_name,
    COUNT(DISTINCT r.respondent_id)               AS n_respondents,
    ROUND(AVG(r.score_normalized)::numeric, 4)    AS norm_score,
    ROUND(MIN(r.score_normalized)::numeric, 4)    AS min_score,
    ROUND(MAX(r.score_normalized)::numeric, 4)    AS max_score,
    ROUND(STDDEV(r.score_normalized)::numeric, 4) AS std_dev
FROM responses r
JOIN projects p ON r.project_id = p.project_id
WHERE r.variable_name IN ('overall liking', 'purchase intent w/o price')
GROUP BY p.category, p.sub_category, r.variable_name
ORDER BY p.category, p.sub_category, r.variable_name;
"""

df_norm = pd.read_sql(query, engine)
df_norm

,category,sub_category,variable_name,n_respondents,norm_score,min_score,max_score,std_dev
0,Beverages,Coffee,overall liking,121,0.8292,0.1667,1.0000,0.1806
1,Beverages,Coffee,purchase intent w/o price,121,0.5220,0.1667,0.6667,0.1343
2,Beverages,Ground Coffee / Tubruk,overall liking,420,0.8583,0.1667,1.0000,0.1428
3,Beverages,Ground Coffee / Tubruk,purchase intent w/o price,420,0.8113,0.2500,1.0000,0.1691
4,Confectionary,Candy,purchase intent w/o price,178,0.7963,0.2500,1.0000,0.1490
5,Food,Biscuit,overall liking,150,0.8589,0.3333,1.0000,0.1334
6,Food,Biscuit,purchase intent w/o price,50,0.8200,0.2500,1.0000,0.1600
7,Food,Noodles,overall liking,307,0.8750,0.3333,1.0000,0.1413
8,Food,Noodles,purchase intent w/o price,247,0.8584,0.1667,1.0000,0.1408
9,Food,Sauce,overall liking,150,0.7033,0.2500,1.0000,0.1771
